In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import inv, matrix_rank, det

print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)

NumPy  : 2.4.2
Pandas : 2.3.3


In [47]:
import numpy as np
from numpy.linalg import inv, matrix_rank


def det_checker(X):
    m, d = X.shape
    if m == d:
        return "even"
    elif m > d:
        return "over"
    else:
        return "under"


def RC_checker(X, y):
    X_aug = np.append(X, y, axis=1)
    rankX = matrix_rank(X)
    rankX_ = matrix_rank(X_aug)
    d = X.shape[1]
    if rankX == rankX_:
        RC = 1 if rankX == d else 3
    else:
        RC = 2
    return RC, rankX, rankX_


def evenSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X) @ y, "Unique solution."
    elif RC == 2:
        return None, "No solution."
    else:
        return None, "Infinitely many solutions."


def overSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X.T @ X) @ X.T @ y, "Unique solution."
    elif RC == 3:
        return None, "Infinitely many solutions."
    else:
        return inv(X.T @ X) @ X.T @ y, "No exact sol, least square approx."


def underSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 2:
        return None, "No solution."
    else:
        return X.T @ inv(X @ X.T) @ y, "No exact sol, least norm approx."


def solveLE(X, y):
    det = det_checker(X)
    if det == "even":
        w, ans = evenSolver(X, y)
    elif det == "over":
        w, ans = overSolver(X, y)
    else:
        w, ans = underSolver(X, y)
    print(ans, "\nw =", w)
    return w


# --- Quick diagnostic ---
X = np.array([[1, 3], [1, 4], [1, 5], [1, 6], [1, 7]])
y = np.array([[5], [4], [3], [2], [1]])

print("System type:", det_checker(X))
RC, rX, rX_ = RC_checker(X, y)
print(f"rank(X)={rX}, rank(X~)={rX_}, case={RC}")
w = solveLE(X, y)

System type: over
rank(X)=2, rank(X~)=2, case=1
Unique solution. 
w = [[ 8.]
 [-1.]]


In [48]:
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import inv
import numpy as np


def polyTx(X, order):
    """Polynomial feature matrix with bias column. Shape: (N, order+1)."""
    return PolynomialFeatures(order).fit_transform(X)


def solvePR(P, y, ridge=False, lamb=0.01):
    """Solve polynomial regression. Auto primal (N>M) or dual (N<M)."""
    if ridge:
        if P.shape[0] > P.shape[1]:  # Primal
            w = inv(P.T @ P + lamb * np.eye(P.shape[1])) @ P.T @ y
        else:  # Dual
            w = P.T @ inv(P @ P.T + lamb * np.eye(P.shape[0])) @ y
    else:
        if P.shape[0] > P.shape[1]:  # Primal
            w = inv(P.T @ P) @ P.T @ y
        else:  # Dual
            w = P.T @ inv(P @ P.T) @ y
    return w


def solveLE_Ridge(X, y, lamb=0.01):
    """Linear regression with Ridge. X must already include bias column."""
    return solvePR(X, y, ridge=True, lamb=lamb)


# Quick test
X_test_poly = np.array([[-10], [-8], [-3], [-1], [2], [8]])
y_test_poly = np.array([[5], [5], [4], [3], [2], [2]])
P = polyTx(X_test_poly, 3)
print("P shape:", P.shape)  # (6, 4): [1, x, x^2, x^3]
w = solvePR(P, y_test_poly)
print("w:", w.ravel())

P shape: (6, 4)
w: [ 2.68935636 -0.37722517  0.01343815  0.00285772]


In [49]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[1], [2], [2.9]])
y = np.array([[2], [4.1], [6.1]])

# --- With bias ---
bias = np.ones((X.shape[0], 1))
X_b = np.hstack((bias, X))  # [1, x]
w = solveLE(X_b, y)  # w = [w0_bias, w1_slope]
w

# # Predict for new inputs
# X_test   = np.array([[30], [5]])
# X_test_b = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
# y_pred   = X_test_b @ w
# print('Predictions:', y_pred.ravel())

# # --- Without bias ---
# w_nb = solveLE(X, y)

# # --- Plot both ---
# plt.scatter(X, y, color='blue', label='Training Samples', marker='o')
# plt.plot(X, X_b @ w,    color='red',   label='With bias')
# plt.plot(X, X   @ w_nb, color='green', label='No bias')
# plt.xlabel('Students'); plt.ylabel('Books sold'); plt.legend(); plt.show()

No exact sol, least square approx. 
w = [[-0.17509225]
 [ 2.15682657]]


array([[-0.17509225],
       [ 2.15682657]])

In [50]:
X = np.array([1, 2, 0, 1, 2, 3]).reshape(3, 2)
print(X)


def rightInverse(X):
    # X: np.array of m x d matrix, m sample, d features for m < d, wide matrix
    # return: np.array of d x m matrix, d features, m sample, right inverse of X

    # to have right inverse, X must be full row rank, rank = m, or XXT is invertible
    # RI = X.T @ inv(X @ X.T)

    if det(X @ X.T) == 0:
        raise ValueError("[rightInverse]: X @ X.T is singular, no right inverse exists")

    return X.T @ inv(X @ X.T)
# rightInverse(X)

[[1 2]
 [0 1]
 [2 3]]


In [51]:
order = 2
X = np.array([12, 25, 7, 16, 30, 3, 14, 22, 10, 10, 28, 5]).reshape(4, 3)
Y = np.array([4.2, 2.5, 3.8, 3.1, 4.5, 4.0, 3.0, 2.8]).reshape(4, 2)
print(X)
print(Y)
P = polyTx(X, order)  # shape (N, 4): [1, x, x^2, x^3]
print(P.shape)
print("NOTICE: PolynomialFeatures column order may differ from handwritten version!")
print("P:\n", P)


X_aug = np.append(X, Y, axis=1)
print("\nrank(X)  =", matrix_rank(X))
print("rank(X~) =", matrix_rank(X_aug))
# Fit
# w_poly = solvePR(P, Y)
# print("Coefficients:", w_poly.ravel())

[[12 25  7]
 [16 30  3]
 [14 22 10]
 [10 28  5]]
[[4.2 2.5]
 [3.8 3.1]
 [4.5 4. ]
 [3.  2.8]]
(4, 10)
NOTICE: PolynomialFeatures column order may differ from handwritten version!
P:
 [[  1.  12.  25.   7. 144. 300.  84. 625. 175.  49.]
 [  1.  16.  30.   3. 256. 480.  48. 900.  90.   9.]
 [  1.  14.  22.  10. 196. 308. 140. 484. 220. 100.]
 [  1.  10.  28.   5. 100. 280.  50. 784. 140.  25.]]

rank(X)  = 3
rank(X~) = 4


## help me


In [52]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[1, 1], [2, 1], [1, 2], [2, 3]])
y = np.array([[2], [3.1], [3.5], [4]])

# --- With bias ---
bias = np.ones((X.shape[0], 1))
X_b = np.hstack((bias, X))  # [1, x]
w = solveLE(X_b, y)  # w = [w0_bias, w1_slope]
print(w[0])

y_full = X_b @ w

from sklearn.metrics import mean_squared_error

MSE = mean_squared_error(y_full, y)
MSE

# # Predict for new inputs
# X_test   = np.array([[30], [5]])
# X_test_b = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
# y_pred   = X_test_b @ w
# print('Predictions:', y_pred.ravel())

# # --- Without bias ---
# w_nb = solveLE(X, y)

# # --- Plot both ---
# plt.scatter(X, y, color='blue', label='Training Samples', marker='o')
# plt.plot(X, X_b @ w,    color='red',   label='With bias')
# plt.plot(X, X   @ w_nb, color='green', label='No bias')
# plt.xlabel('Students'); plt.ylabel('Books sold'); plt.legend(); plt.show()

No exact sol, least square approx. 
w = [[1.29]
 [0.47]
 [0.66]]
[1.29]


0.11025000000000001

## help pls

In [53]:
import numpy as np
from numpy.linalg import inv

X = np.array([[1, 1], [2, 1], [1, 2], [2, 3]])
y = np.array([[2], [3.1], [3.5], [4]])
order = 2
P = polyTx(X, order)

# --- Via tool (auto primal/dual) ---
lamb = 0.1  # 'lambda' is a Python keyword
w_ridge = solvePR(P, y, ridge=True, lamb=lamb)
print("Ridge w (1dp):", np.around(w_ridge.T, decimals=4))
y_full = P @ w_ridge

from sklearn.metrics import mean_squared_error

MSE = mean_squared_error(y_full, y)
MSE

Ridge w (1dp): [[ 0.6502  0.5576  1.1442  0.3723 -0.8213  0.2593]]


0.02456869669872603

### helppp

In [55]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import inv

X = np.array([[2, 1, 0], [0, 3, 1], [1, 0, 3], [3, 1, 4], [-1, 2, 1]])
y = np.array([[1, 0, 0], [0, 0, 1], [0, 1, 0], [1, 0, 0], [0, 1, 0]])
order = 2
P = polyTx(X, order)

# --- Via tool (auto primal/dual) ---
lamb = 0.01  # 'lambda' is a Python keyword
w_hot = solvePR(P, y, ridge=True, lamb=lamb)
print(w_hot.shape)
print(w_hot[0][1])
# Predict on new point
x_test = np.array([[1, 1, 2]])
P_test = polyTx(x_test, order)  # same order
y_pred_poly = P_test @ w_hot
y_pred_poly  

(10, 3)
0.11092343297230106


array([[0.16106689, 0.19636178, 0.17533303]])